In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import joblib
import sys, sklearn, numpy

In [ ]:
df = pd.read_csv("Banking_Transactions_USA_2023_2024.csv")



In [3]:
# Convert Fraud_Flag, Transaction_Status to 1 and 0
df['Fraud_Flag'] = df['Fraud_Flag'].map({'Yes': 1, 'No': 0})

df['Transaction_Status'] = df['Transaction_Status'].map({'Success': 1, 'Failed': 0, 'Pending': 0})


# Convert Transaction_Date to datetime
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

# Convert Transaction_Date to numeric features
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['hour'] = df['Transaction_Date'].dt.hour
df['day_of_week'] = df['Transaction_Date'].dt.dayofweek
df['month'] = df['Transaction_Date'].dt.month






In [4]:
# 2. Features and target
# -------------------------------
features = [
    'hour',
    'day_of_week',
    'month',
    'Transaction_Amount',
    'Transaction_Type',
    'Category',
    'City',
    'Country',
    'Payment_Method',
    'Customer_Age',
    'Customer_Gender',
    'Customer_Occupation',
    'Account_Balance',
    'Transaction_Status'
]
#Remove id and target column
target = 'Fraud_Flag'

X = df[features]
y = df[target]




In [5]:
# Numeric columns
num_features = ['Transaction_Amount', 'Customer_Age', 'Account_Balance',
                'Transaction_Status', 'hour', 'day_of_week', 'month']

# Categorical columns
cat_features = ['Transaction_Type', 'Category', 'City', 'Country', 'Payment_Method',
                'Customer_Gender', 'Customer_Occupation']

# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

In [6]:
# 4. Train/test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)


In [7]:
# 5. Model pipeline
# -------------------------------
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced',  # handle imbalance
        random_state=42
    ))
])

In [8]:
# 6. Train model
# -------------------------------
pipeline.fit(X_train, y_train)

# -------------------------------
# 7. Evaluate model
# -------------------------------
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.53      0.56      0.55       820
           1       0.52      0.49      0.50       797

    accuracy                           0.53      1617
   macro avg       0.53      0.53      0.53      1617
weighted avg       0.53      0.53      0.53      1617

ROC AUC: 0.517417449582275


In [9]:
# 8. Save model for later use
# -------------------------------
joblib.dump(pipeline, "fraud_model_1.pkl")

['fraud_model_1.pkl']